In [12]:
# Celda 1 Librerias y el modelo
import json 
import sys
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC, FIG = ROOT / "data/processed", ROOT / "figures"
PROC.mkdir(parents=True, exist_ok=True)
FIG.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))
import estilo as E
from modelo_dsi import Parametros, condicionales, medir_dsi, simular

E.aplicar_estilo()
pd.set_option("display.width", 140)

In [13]:
# Celda 2 Los datos contra los que se mide el modelo (CA1 de raton, pulso de 5 s, un IPSC cada 4 s)

datos = pd.DataFrame([
    ("Control", 32.8, 3.9, 9.5, 1.8, "Pan et al. 2011"),
    ("Sin MAGL (cronico)", 20.5, 3.2, 25.6, 5.5, "Pan et al. 2011"),
    ("Sin DAGLα", -2.3, 1.0, np.nan, np.nan, "Gao et al. 2010"),
], columns=["condicion", "magnitud", "error_mag", "tau", "error_tau", "fuente"]).set_index("condicion")

# El agonista WIN 55,212-2 (2 µM) deprime el IPSC 41.3% en controles y 24.3% sin MAGL:
# Sin MAGL queda el 59% de los CB1 funcionales (desensibilizacion por exceso de 2-AG).
F_CB1_SIN_MAGL = 24.3 / 41.3
datos

,magnitud,error_mag,tau,error_tau,fuente
condicion,,,,,
Control,32.8,3.9,9.5,1.8,Pan et al. 2011
Sin MAGL (cronico),20.5,3.2,25.6,5.5,Pan et al. 2011
Sin DAGLα,-2.3,1.0,NaN,NaN,Gao et al. 2010


In [16]:
# Celda 3 · Una DSI con los parámetros de arranque: qué hace cada pieza
p0 = Parametros()
sim = simular(p0)
fig, ejes = plt.subplots(3, 1, figsize=(7, 6), sharex=True)
for ax, clave, etiqueta in zip(ejes, ["ca", "2ag", "ipsc"], ["Ca²⁺ (µM)", "2-AG (× Kd)", "IPSC relativo"]):
    ax.axvspan(0, 5, color=E.REJILLA, lw=0)
    ax.plot(sim["t"], sim[clave], color=E.AZUL)
    ax.set_ylabel(etiqueta)
ejes[-1].set_xlabel("Tiempo desde el inicio del pulso (s)")
ejes[0].set_title("El 2-AG dura más que el Ca²⁺ que lo produjo")
plt.show()
m = medir_dsi(sim)
print(f"Con los parámetros de arranque: magnitud {m['magnitud']:.1f} %, tau {m['tau']:.1f} s")

TypeError: 'tuple' object is not callable